In [ ]:
import pandas as pd
import numpy as np

# ================================================================= #
# PART A, STEP 1: LOAD DATASETS & DOCUMENT                          #
# ================================================================= #

trader_df = pd.read_csv("historical_data.csv")
sentiment_df = pd.read_csv("fear_greed_index.csv")

# ================================================================= #
# CALCULATE SHAPE, MISSING VALUES, AND DUPLICATES                   #
# ================================================================= #

print("--- Trader Data ---")
print(f"Rows: {trader_df.shape[0]}, Columns: {trader_df.shape[1]}")
print(f"Missing Values: {trader_df.isnull().sum().sum()}")
print(f"Duplicates: {trader_df.duplicated().sum()}")

print("\n--- Sentiment Data ---")
print(f"Rows: {sentiment_df.shape[0]}, Columns: {sentiment_df.shape[1]}")
print(f"Missing Values: {sentiment_df.isnull().sum().sum()}")
print(f"Duplicates: {sentiment_df.duplicated().sum()}")

--- Trader Data ---
Rows: 46482, Columns: 16
Missing Values: 5
Duplicates: 0

--- Sentiment Data ---
Rows: 2644, Columns: 4
Missing Values: 0
Duplicates: 0


/tmp/ipykernel_4613/2324808887.py:8: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  trader_df = pd.read_csv("historical_data.csv")


In [ ]:
# ================================================================= #
# PART A, STEP 2: CONVERT TIMESTAMPS & ALIGN DATASETS               #
# ================================================================= #

# 1. Clean the 5 missing values found in the previous step
trader_df.dropna(inplace=True)

# 2. Convert timestamps to a standard 'YYYY-MM-DD' daily string
# Using errors='coerce' and dayfirst=True prevents formatting crashes
trader_df['Date'] = pd.to_datetime(trader_df['Timestamp IST'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')
sentiment_df['Date'] = pd.to_datetime(sentiment_df['date']).dt.strftime('%Y-%m-%d')

# 3. Merge datasets based on the Date (Left Join)
merged_df = pd.merge(trader_df, sentiment_df[['Date', 'classification', 'value']], on='Date', how='left')

# Drop any outlier trades where sentiment data wasn't available for that day
merged_df.dropna(subset=['classification'], inplace=True)
print(f"✅ Merged Dataset Ready! Shape: {merged_df.shape}")

# ================================================================= #
# PART A, STEP 3: CREATE KEY METRICS                                #
# ================================================================= #

# Helper column: 1 if it's a winning trade, 0 if it's a losing trade
merged_df['Is_Win'] = (merged_df['Closed PnL'] > 0).astype(int)

# Group the data to calculate daily PnL, win rate, and average trade size per account
daily_metrics = merged_df.groupby(['Account', 'Date']).agg(
    Daily_PnL=('Closed PnL', 'sum'),
    Total_Trades=('Closed PnL', 'count'),
    Winning_Trades=('Is_Win', 'sum'),
    Avg_Trade_Size_USD=('Size USD', 'mean')
).reset_index()

# Calculate the Win Rate decimal
daily_metrics['Win_Rate'] = daily_metrics['Winning_Trades'] / daily_metrics['Total_Trades']

print("✅ Daily Metrics Successfully Engineered!")
daily_metrics.head(3)

✅ Merged Dataset Ready! Shape: (46475, 19)
✅ Daily Metrics Successfully Engineered!


,Account,Date,Daily_PnL,Total_Trades,Winning_Trades,Avg_Trade_Size_USD,Win_Rate
0,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-11,0.0,177,0,5089.718249,0.0
1,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-17,0.0,68,0,7976.664412,0.0
2,0x083384f897ee0f19899168e3b1bec365f52a9012,2024-11-18,0.0,40,0,23734.500000,0.0


In [ ]:
# @title
import plotly.express as px

# ================================================================= #
# PART B: DATA ANALYSIS & INSIGHTS                                  #
# ================================================================= #

print("--- 1. Performance by Market Sentiment ---")
# Calculate Win Rate and Average PnL for each Sentiment Classification
performance = merged_df.groupby('classification').agg(
    Total_Trades=('Closed PnL', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Win_Rate=('Is_Win', lambda x: x.sum() / len(x))
).reset_index()
print(performance.to_string(index=False))

print("\n--- 2. Trader Behavior by Market Sentiment ---")
# Calculate Average Position Size and what % of trades are BUY (Long Ratio)
behavior = merged_df.groupby('classification').agg(
    Avg_Trade_Size_USD=('Size USD', 'mean'),
    Long_Ratio=('Side', lambda x: (x == 'BUY').sum() / len(x))
).reset_index()
print(behavior.to_string(index=False))

print("\n--- 3. Trader Segmentation (Frequent vs Infrequent) ---")
# Classify traders with >100 trades as 'Frequent', else 'Infrequent'
trade_counts = merged_df['Account'].value_counts()
freq_traders = trade_counts[trade_counts > 100].index
merged_df['Segment'] = np.where(merged_df['Account'].isin(freq_traders), 'Frequent', 'Infrequent')

# Check how segments perform during different market sentiments
segment_perf = merged_df.groupby(['Segment', 'classification'])['Closed PnL'].mean().reset_index()
print(segment_perf.head()) # Previewing the segment table

# ================================================================= #
# PART B: VISUALIZATIONS (PLOTLY)                                   #
# ================================================================= #

# Chart 1: Average PnL by Sentiment
fig1 = px.bar(
    performance,
    x='classification',
    y='Avg_PnL',
    title="Average PnL per Trade by Market Sentiment",
    color='classification',
    text_auto='.2f'
)
fig1.show()

# Chart 2: Buy (Long) Ratio by Sentiment
fig2 = px.bar(
    behavior,
    x='classification',
    y='Long_Ratio',
    title="Long (BUY) Bias by Market Sentiment",
    color='classification',
    text_auto='.1%'
)
fig2.show()

--- 1. Performance by Market Sentiment ---
classification  Total_Trades    Avg_PnL  Win_Rate
  Extreme Fear          1746 204.336799  0.404926
 Extreme Greed          9354 162.759181  0.536883
          Fear         12114 146.672879  0.445022
         Greed         15166  92.228518  0.411315
       Neutral          8095  86.191512  0.479061

--- 2. Trader Behavior by Market Sentiment ---
classification  Avg_Trade_Size_USD  Long_Ratio
  Extreme Fear         9791.737211    0.569301
 Extreme Greed         8168.774757    0.452641
          Fear        23119.389312    0.528314
         Greed        14226.886005    0.483780
       Neutral        13987.707452    0.508833

--- 3. Trader Segmentation (Frequent vs Infrequent) ---
    Segment classification  Closed PnL
0  Frequent   Extreme Fear  204.336799
1  Frequent  Extreme Greed  162.759181
2  Frequent           Fear  146.672879
3  Frequent          Greed   92.228518
4  Frequent        Neutral   86.191512


In [1]:
import pandas as pd

# Load the files
trader_df = pd.read_csv("historical_data.zip", low_memory=False)
sentiment_df = pd.read_csv("fear_greed_index.csv")

# Print the actual column names so we can stop guessing
print("TRADER COLUMNS:")
print(trader_df.columns.tolist())
print("\nSENTIMENT COLUMNS:")
print(sentiment_df.columns.tolist())

TRADER COLUMNS:
['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']

SENTIMENT COLUMNS:
['timestamp', 'value', 'classification', 'date']


In [4]:
import pandas as pd

# ==============================================================================
# 1. LOAD THE DATA
# ==============================================================================
trader_df = pd.read_csv("historical_data.zip", low_memory=False)
sentiment_df = pd.read_csv("fear_greed_index.csv")

# ==============================================================================
# 2. STANDARDIZE THE TIME COLUMNS (FIXED)
# Target 'Timestamp IST' instead of the raw Unix 'Timestamp' to prevent
# 1970s epoch parsing errors and ensure dates overlap perfectly.
# ==============================================================================
trader_df['Date_Match'] = pd.to_datetime(trader_df['Timestamp IST'], errors='coerce').dt.date
sentiment_df['Date_Match'] = pd.to_datetime(sentiment_df['date'], errors='coerce').dt.date

# ==============================================================================
# 3. MERGE THE DATASETS
# ==============================================================================
merged_df = pd.merge(trader_df, sentiment_df, on='Date_Match', how='inner')

print(f"SUCCESS! Data merged perfectly. Final shape: {merged_df.shape}")

SUCCESS! Data merged perfectly. Final shape: (35864, 21)


In [5]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

# Group data by Account to get individual trader stats
trader_stats = merged_df.groupby('Account').agg(
    Total_Trades=('Closed PnL', 'count'),
    Avg_PnL=('Closed PnL', 'mean'),
    Avg_Size=('Size USD', 'mean')
).fillna(0)

# Scale the data for clustering
scaler = StandardScaler()
scaled_stats = scaler.fit_transform(trader_stats)

# Run K-Means to find 3 behavioral archetypes
kmeans = KMeans(n_clusters=3, random_state=42)
trader_stats['Archetype'] = kmeans.fit_predict(scaled_stats)

# Map numeric clusters to readable names
archetype_map = {
    0: "Retail / Small Size Traders",
    1: "High-Frequency Grinders",
    2: "Whales / High Capital Traders"
}
trader_stats['Archetype_Name'] = trader_stats['Archetype'].map(archetype_map)

print("Trader Archetypes Discovered:")
print(trader_stats['Archetype_Name'].value_counts())

Trader Archetypes Discovered:
Archetype_Name
Retail / Small Size Traders      28
Whales / High Capital Traders     2
High-Frequency Grinders           2
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Prepare the Machine Learning DataFrame
ml_df = merged_df.dropna(subset=['classification', 'Size USD', 'Side', 'Closed PnL']).copy()

# Feature Engineering: Create the target variable 'Is_Win' (True if profit is positive)
ml_df['Is_Win'] = ml_df['Closed PnL'] > 0

# Convert text categories into numbers for the model
ml_df['Side_Numeric'] = ml_df['Side'].apply(lambda x: 1 if str(x).lower() == 'buy' else 0)
sentiment_map = {'Extreme Fear': 0, 'Fear': 1, 'Neutral': 2, 'Greed': 3, 'Extreme Greed': 4}
ml_df['Sentiment_Numeric'] = ml_df['classification'].map(sentiment_map)

# Drop any rows where sentiment mapping failed (just to be safe)
ml_df = ml_df.dropna(subset=['Sentiment_Numeric'])

# Define Features (X) and Target (y)
X = ml_df[['Size USD', 'Side_Numeric', 'Sentiment_Numeric']]
y = ml_df['Is_Win']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a lightweight Random Forest model
clf = RandomForestClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# Predict and evaluate
preds = clf.predict(X_test)
print("Predictive Model Accuracy:", round(accuracy_score(y_test, preds) * 100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, preds))

Predictive Model Accuracy: 64.92 %

Classification Report:
               precision    recall  f1-score   support

       False       0.67      0.76      0.71      4087
        True       0.61      0.50      0.55      3086

    accuracy                           0.65      7173
   macro avg       0.64      0.63      0.63      7173
weighted avg       0.64      0.65      0.64      7173

